In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd

X_train = pd.read_csv("../src/Dataset/X_train.csv").to_numpy()
X_test = pd.read_csv("../src/Dataset/X_test.csv").to_numpy()
y_train = pd.read_csv("../src/Dataset/y_train.csv").to_numpy().ravel()
y_test = pd.read_csv("../src/Dataset/y_test.csv").to_numpy().ravel()


# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Healthy', 'Low Risk', 'Injured']))

              precision    recall  f1-score   support

     Healthy       0.68      0.96      0.79      2961
    Low Risk       0.00      0.00      0.00       971
     Injured       0.59      0.36      0.45       694

    accuracy                           0.67      4626
   macro avg       0.42      0.44      0.41      4626
weighted avg       0.52      0.67      0.57      4626



In [3]:
from sklearn.metrics import brier_score_loss, roc_auc_score

# Probability columns follow model.classes_: Healthy (0), Low Risk (1), Injured (2).
y_proba = model.predict_proba(X_test)

# Multiclass Brier score: sum of squared errors across classes, averaged over samples.
# Range: 0 to 2; lower is better.
brier = brier_score_loss(
    y_test, y_proba, labels=model.classes_, scale_by_half=False
)

# Treat each class as positive versus all other classes, then average equally.
# Higher is better; 0.5 indicates chance-level ranking.
roc_auc = roc_auc_score(
    y_test, y_proba, labels=model.classes_, multi_class="ovr", average="macro"
)

print(f"Multiclass Brier score (0-2, lower is better): {brier:.4f}")
print(f"Macro ROC AUC (one-vs-rest, higher is better): {roc_auc:.4f}")


Multiclass Brier score (0-2, lower is better): 0.4690
Macro ROC AUC (one-vs-rest, higher is better): 0.7022


In [4]:
from sklearn.metrics import log_loss

# predict_proba returns one probability per class for each test sample.
y_proba = model.predict_proba(X_test)
class_names = {0: "Healthy", 1: "Low Risk", 2: "Injured"}
probability_predictions = pd.DataFrame(
    y_proba,
    columns=[f"P({class_names[label]})" for label in model.classes_],
)
probability_predictions.insert(0, "Actual", pd.Series(y_test).map(class_names))
probability_predictions.insert(1, "Predicted", pd.Series(y_pred.ravel()).map(class_names))
print("Probability predictions (first 5 test samples):")
print(probability_predictions.head().to_string(index=False, float_format="{:.4f}".format))

# Evaluate each class against all other classes using probabilities, not hard labels.
rows = []
for column, label in enumerate(model.classes_):
    is_class = (y_test == label).astype(int)
    probabilities = y_proba[:, column]
    rows.append({
        "Class": class_names[label],
        "Support": int(is_class.sum()),
        "Actual proportion": is_class.mean(),
        "Mean predicted probability": probabilities.mean(),
        "Brier (0-1)": brier_score_loss(is_class, probabilities, scale_by_half=True),
        "ROC AUC (OvR)": roc_auc_score(is_class, probabilities),
    })
probability_report = pd.DataFrame(rows).set_index("Class")
print("\nPer-class probability report:")
print(probability_report.to_string(float_format="{:.4f}".format))
print(f"\nMulticlass log loss (lower is better): {log_loss(y_test, y_proba, labels=model.classes_):.4f}")
print("Brier: lower is better. ROC AUC: higher is better (0.5 is chance-level ranking).")
# The multiclass Brier score above equals the sum of the per-class Brier scores.


Probability predictions (first 5 test samples):
  Actual Predicted  P(Healthy)  P(Low Risk)  P(Injured)
 Injured   Healthy      0.4300       0.2100      0.3600
 Healthy   Healthy      0.4400       0.1600      0.4000
Low Risk   Healthy      0.7300       0.2600      0.0100
 Injured   Healthy      0.6800       0.0800      0.2400
Low Risk   Healthy      0.5600       0.2700      0.1700

Per-class probability report:
          Support  Actual proportion  Mean predicted probability  Brier (0-1)  ROC AUC (OvR)
Class                                                                                       
Healthy      2961             0.6401                      0.6242       0.2121         0.6516
Low Risk      971             0.2099                      0.2247       0.1655         0.5698
Injured       694             0.1500                      0.1511       0.0914         0.8853

Multiclass log loss (lower is better): 0.7698
Brier: lower is better. ROC AUC: higher is better (0.5 is chance-level ra